# Tamizaje sintético de Revisor-CRF en Google Colab

Esta libreta usa únicamente los 30 casos sintéticos versionados. **No suba CRF, Excel, bancos reales, carpetas `results/` ni información institucional.** Un resultado favorable sólo significa `synthetic_screen_passed`; nunca aprueba el modelo para uso institucional.

Antes de ejecutar, seleccione **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.

In [ ]:
import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True, capture_output=True, check=False,
)
if gpu.returncode != 0:
    raise RuntimeError("No hay GPU asignada. Cambie el tipo de entorno a GPU y reconecte.")
print(gpu.stdout.strip())

## Obtener una copia limpia del código

La libreta no conecta Google Drive ni solicita archivos locales.

In [ ]:
import pathlib
import sys

repo = pathlib.Path("/content/revisor-crf")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/arodriguezg2106/PROYECTO-IA-ASF.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo)], check=True)

## Instalar Ollama y descargar el primer candidato

`qwen3.5:9b` ocupa aproximadamente 6.6–7.6 GB según el catálogo de Ollama. La disponibilidad y el tipo de GPU de Colab pueden cambiar.

In [ ]:
import os
import time
import urllib.request

subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
log = open("/tmp/ollama.log", "w")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama no inició; revise /tmp/ollama.log")
subprocess.run(["ollama", "pull", "qwen3.5:9b"], check=True)

## Ejecutar las tres compuertas sintéticas

Se ejecutan 2 casos de humo, 10 difíciles y 30 finales, cada uno tres veces. No se carga ni se simula la compuerta del banco real.

In [ ]:
output = pathlib.Path("/content/qwen35_9b_synthetic.json")
environment = os.environ.copy()
environment["PYTHONPATH"] = f"{repo / 'src'}:{repo / 'scripts'}"
command = [
    sys.executable, str(repo / "scripts/evaluate_editorial_context.py"),
    "--synthetic-only", "--models", "qwen3.5:9b",
    "--available-models", "qwen3.5:9b", "--repetitions", "3",
    "--dataset", str(repo / "tests/data/editorial_context_cases.json"),
    "--config", str(repo / "editorial-config.example.toml"),
    "--output", str(output),
]
completed = subprocess.run(command, cwd=repo, env=environment, text=True)
print("Código de salida:", completed.returncode)
if not output.exists():
    raise RuntimeError("La evaluación no generó el reporte.")

In [ ]:
import json

report = json.loads(output.read_text(encoding="utf-8"))
print("Alcance:", report["evaluation_scope"])
print("Estado:", report["status"])
print("Modelo tamizado:", report["screened_model"])
print("Modelo aprobado:", report["selected_model"])
for result in report["results"]:
    for stage in result.get("stages", []):
        metrics = stage.get("metrics", {})
        print(stage["stage"], stage["status"], {key: metrics.get(key) for key in ("precision", "recall", "accuracy", "prediction_agreement")})

Descargue únicamente el JSON agregado si necesita conservar la evidencia. No añada textos o archivos institucionales a esta sesión.

In [ ]:
from google.colab import files
files.download(str(output))